# Wang et al 2021 Data Explorer

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets

COLORS = {
    'electron': "#1f26b4", 
    'proton': "#d60303"
}
MeV_MARKERS = {1: 'o', 3: 's', 10: 'd'}  # by energy (MeV)


In [ ]:
df = pd.read_csv("./Wang2021.csv")
df.columns = [c.strip() for c in df.columns]
print("Rows:", len(df))
print("Materials:", sorted(df['material'].unique()))
print("Particle types:", sorted(df['particle type'].unique()))
print("Energies (MeV):", sorted(df['energy (MeV)'].unique()))

In [ ]:
output_vars = ['P_max', 'V_oc', 'I_SC', 'FF']

y_w = widgets.Dropdown(
    options=output_vars,
    value='P_max',
    description='Y-axis:',
)

def plot_1(
    param: str
):

    fig, ax = plt.subplots(figsize=(7, 5))
    sub = df.copy()
    for energy, grp2 in sub.groupby('energy (MeV)'):
        grp2 = grp2.sort_values('fluence (e/cm^2)')
        plt.plot(
            grp2['fluence (e/cm^2)'], 
            grp2[param],
            marker=MeV_MARKERS.get(energy, 'o'),
            color=COLORS.get('electron', 'gray'),
            linestyle='-',
            label=f"electron, {energy} MeV",
        )
    plt.legend()
    plt.grid(ls='--')
    plt.xscale('log')
    plt.xlabel('Fluence (e/cm^2)')
    plt.ylabel(f'{param}/{param}_0')
    plt.title('GaInP/GaAs/Ge resilience')

ui = widgets.HBox([y_w])

out = widgets.interactive_output(
    plot_1,
    {
        'param': y_w
    }
)
display(ui, out)

In [ ]:
y1_w = widgets.Dropdown(
    options=output_vars, 
    value='P_max', 
    description='Param 1:'
)
y2_w = widgets.Dropdown(
    options=output_vars, 
    value='I_SC', 
    description='Param 2:'
)

def plot_compare(y1, y2):
    _, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, y_col in zip(axes, [y1, y2]):
        sub = df.dropna(subset=[y_col])
        for (ptype, energy), grp in sub.groupby(['particle type', 'energy (MeV)']):
            grp = grp.sort_values('fluence (e/cm^2)')
            ax.plot(
                grp['fluence (e/cm^2)'], 
                grp[y_col],
                marker=MeV_MARKERS.get(energy, 'o'),
                color=COLORS.get(ptype, 'gray'),
                label=f"{ptype}, {energy} MeV",
            )

        ax.set_xlabel('Fluence (cm$^{-2}$)')
        ax.set_ylabel(y_col)
        ax.set_title(y_col)

        if (sub['fluence (e/cm^2)'] <= 0).any():
            ax.set_xscale('symlog')
        else:
            ax.set_xscale('log')

        ax.grid(ls='--')
        ax.legend(fontsize=7)
        ax.set_ylim(0.35, 1.05)

    plt.tight_layout()
    plt.show()


compare_out = widgets.interactive_output(
    plot_compare,
    {
        'y1': y1_w, 
        'y2': y2_w
    }
)
display(widgets.HBox([y1_w, y2_w]), compare_out)